# 01 — Data Preparation + Gemma 4 E2B-it Baseline Evaluation

Downloads PubMedQA and MedQA datasets, formats them for fine-tuning using the
Gemma chat prompt template, and runs baseline evaluation on the unmodified
`google/gemma-4-E2B-it` model loaded via UnSloth (4-bit NF4 on T4/A100).

**Hardware target**: Kaggle 2×T4 (16 GB each) or any single GPU ≥ 8 GB VRAM.

In [ ]:
# Install all dependencies
# UnSloth ≥0.1.36 is required for Gemma 4 E2B support
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers>=4.50.0" datasets accelerate sentencepiece \
    protobuf bitsandbytes trl pandas tqdm scipy

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"VRAM GPU0: {torch.cuda.mem_get_info(0)[1] / 1024**3:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"VRAM GPU1: {torch.cuda.mem_get_info(1)[1] / 1024**3:.1f} GB")

## 1. Load and Explore Datasets

In [ ]:
from datasets import load_dataset

# PubMedQA — labeled subset, yes/no/maybe classification
pubmedqa = load_dataset("pubmed_qa", "pqa_labeled")
print("PubMedQA splits:", {k: len(v) for k, v in pubmedqa.items()})
print("\nSample:")
print(pubmedqa["train"][0])

In [ ]:
# MedQA — USMLE-style 4-option multiple choice
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")
print("MedQA splits:", {k: len(v) for k, v in medqa.items()})
print("\nSample:")
print(medqa["train"][0])

## 2. Format for Training

Gemma 4 E2B-it uses the same conversation tokens as Gemma 2:
```
<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>
```
We format both datasets into this template so the same training pipeline works.

In [ ]:
PUBMEDQA_TEMPLATE = """<start_of_turn>user
Context: {context}

Question: {question}
Answer with yes, no, or maybe and explain your reasoning.<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>"""

MEDQA_TEMPLATE = """<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>"""


def format_pubmedqa(example):
    ctx_data = example.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context_str = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    long_answer = example.get("long_answer", "")
    final_decision = example.get("final_decision", "")
    answer = f"{final_decision}. {long_answer}" if long_answer else final_decision
    return {"text": PUBMEDQA_TEMPLATE.format(
        context=context_str, question=example["question"], answer=answer
    )}


def format_medqa(example):
    question = example["question"]
    options = example.get("options", {})
    if isinstance(options, dict):
        opts_str = "\n".join(f"  {k}) {v}" for k, v in sorted(options.items()))
        question = f"{question}\n{opts_str}"
    answer_idx = example.get("answer_idx", "")
    answer = example.get("answer", "")
    full_answer = f"The answer is {answer_idx}) {answer}" if answer_idx else answer
    return {"text": MEDQA_TEMPLATE.format(question=question, answer=full_answer)}


# Apply formatting
pubmedqa_fmt = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa["train"].column_names)
medqa_fmt = medqa.map(format_medqa, remove_columns=medqa["train"].column_names)

print("Formatted PubMedQA sample:")
print(pubmedqa_fmt["train"][0]["text"][:600])
print("\n---\n")
print("Formatted MedQA sample:")
print(medqa_fmt["train"][0]["text"][:400])

In [ ]:
from datasets import concatenate_datasets
import os

# PubMedQA has only a 'train' split (1000 examples).
# We use MedQA 'test' as the validation split.
train_dataset = concatenate_datasets([pubmedqa_fmt["train"], medqa_fmt["train"]])
val_dataset = medqa_fmt["test"]

print(f"Combined train : {len(train_dataset)} examples")
print(f"Validation     : {len(val_dataset)} examples")

os.makedirs("data/train", exist_ok=True)
os.makedirs("data/val", exist_ok=True)
train_dataset.save_to_disk("data/train")
val_dataset.save_to_disk("data/val")
print("\nSaved to data/train and data/val")

## 3. Baseline Evaluation — Gemma 4 E2B-it (4-bit via UnSloth)

Load the unmodified instruction-tuned model with UnSloth's `FastModel` to
establish control numbers. 4-bit NF4 keeps VRAM well under the T4 16 GB limit.

> **Note**: UnSloth ≥ 0.1.36 is required for Gemma 4 E2B. The model is
> multimodal (text/image/audio) but we use it text-only here.

In [ ]:
from unsloth import FastModel
import torch

MODEL_NAME = "unsloth/gemma-4-E2B-it"

print(f"Loading {MODEL_NAME} in 4-bit via UnSloth...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,          # auto-detect: bfloat16 if supported, float16 on T4
    full_finetuning=False,
)

# Switch to optimised inference mode (no gradient tracking)
FastModel.for_inference(model)

print(f"\nModel loaded.")
print(f"Parameters     : {model.num_parameters() / 1e9:.2f}B")
print(f"Peak VRAM used : {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# Quick sanity-check generation
device = "cuda"
prompt = "<start_of_turn>user\nWhat are the main symptoms of Type 2 diabetes?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
import time
import gc
from tqdm import tqdm
import pandas as pd

device = "cuda"
results = {"model": "gemma4-e2b-baseline"}
BATCH_SIZE = 2   # conservative for T4 with 4-bit

# ── [1/4] WikiText-2 perplexity ────────────────────────────────────────────
print("[1/4] WikiText-2 perplexity...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:256]

model.eval()
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE)):
    batch = wiki_texts[i : i + BATCH_SIZE]
    enc = tokenizer(
        batch, return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
    del enc, out
torch.cuda.empty_cache()
gc.collect()

results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {results['perplexity_wikitext']:.2f}")

In [ ]:
# ── [2/4] Medical text perplexity ─────────────────────────────────────────
print("[2/4] Medical text perplexity...")
pqa_eval = load_dataset("pubmed_qa", "pqa_labeled", split="train")
med_texts = []
for ex in pqa_eval:
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    ctx = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    if len(ctx.strip()) > 50:
        med_texts.append(ctx)
    if len(med_texts) >= 256:
        break

total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(med_texts), BATCH_SIZE)):
    batch = med_texts[i : i + BATCH_SIZE]
    enc = tokenizer(
        batch, return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
    del enc, out
torch.cuda.empty_cache()
gc.collect()

results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {results['perplexity_medical']:.2f}")

In [ ]:
# ── [3/4] PubMedQA accuracy ───────────────────────────────────────────────
print("[3/4] PubMedQA accuracy...")
correct, total = 0, 0
MAX_SAMPLES = 200

for ex in tqdm(pqa_eval, total=MAX_SAMPLES):
    if total >= MAX_SAMPLES:
        break
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    # Gemma 4 chat format — model continues after <start_of_turn>model
    prompt = (
        f"<start_of_turn>user\n"
        f"Context: {context}\n\n"
        f"Question: {ex['question']}\n"
        f"Answer with exactly one word: yes, no, or maybe.<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=768
    ).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    resp = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip().lower()
    pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
    gold = ex["final_decision"].lower().strip()
    if pred in ("yes", "no", "maybe") and pred == gold:
        correct += 1
    total += 1
    del inputs, out
    torch.cuda.empty_cache()

results["pubmedqa_accuracy"] = correct / total if total else 0
print(f"  -> {results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

In [ ]:
# ── [4/4] Inference speed & peak VRAM ─────────────────────────────────────
print("[4/4] Inference speed & VRAM...")
prompt = "<start_of_turn>user\nWhat is diabetes?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Warmup
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=50, do_sample=False)

torch.cuda.reset_peak_memory_stats()
total_tok, total_t = 0, 0.0
for _ in range(5):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    torch.cuda.synchronize()
    total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    total_t += time.perf_counter() - t0

results["tokens_per_sec"] = total_tok / total_t
results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / 1024**3
print(f"  -> {results['tokens_per_sec']:.1f} tok/s")
print(f"  -> {results['peak_vram_gb']:.2f} GB peak VRAM")

In [ ]:
# Save baseline results
os.makedirs("results/tables", exist_ok=True)
df = pd.DataFrame([results])
df.to_csv("results/tables/all_results.csv", index=False)
print("\nBaseline results:")
df

In [ ]:
# Cleanup before next notebook
del model
torch.cuda.empty_cache()
gc.collect()
print("Done! Proceed to notebook 02 for UnSloth fine-tuning.")